In [1]:
# =========================================
# 🛒 Suryam Retail → STAR SCHEMA PIPELINE
# SQL → Python → Power BI Ready Data Model
# =========================================

import mysql.connector
import pandas as pd
import os

# 📁 Output folder
output_path = "powerbi_data"
os.makedirs(output_path, exist_ok=True)

# =========================================
# 🔌 DATABASE CONNECTION
# =========================================
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="Diksha@2407",
        database="RetailDB"
    )
    print("✅ Connected to MySQL")
except Exception as e:
    print("❌ Connection Error:", e)
    conn = None

# =========================================
# 📦 1. EXPORT RAW TABLES (ALL 8)
# =========================================
tables = ["Supplier","Product","Inventory","Sales",
          "Sales_Details","Customer","Category","Purchase_Pattern"]

for t in tables:
    try:
        df = pd.read_sql(f"SELECT * FROM {t}", conn)
        df.to_csv(f"{output_path}/{t}.csv", index=False)
        print(f"✅ {t}.csv")
    except Exception as e:
        print(f"❌ {t} Error:", e)

# =========================================
# 🌟 2. CREATE FACT TABLE (FACT SALES)
# =========================================
try:
    fact_sales = pd.read_sql("""
        SELECT 
            sd.sale_id,
            s.sale_date,
            s.customer_id,
            sd.product_id,
            sd.quantity,
            sd.total_price
        FROM Sales_Details sd
        JOIN Sales s ON sd.sale_id = s.sale_id
    """, conn)

    fact_sales['sale_date'] = pd.to_datetime(fact_sales['sale_date'])
    fact_sales.to_csv(f"{output_path}/fact_sales.csv", index=False)

    print("✅ fact_sales.csv")

except Exception as e:
    print("❌ Fact Sales Error:", e)

# =========================================
# 🧩 3. DIMENSION TABLES (FIXED)
# =========================================

# 🟣 Product Dimension (NO CHANGE)
try:
    dim_product = pd.read_sql("""
        SELECT p.product_id, p.product_name, p.price,
               c.category_name, s.supplier_name
        FROM Product p
        JOIN Category c ON p.category_id = c.category_id
        JOIN Supplier s ON p.supplier_id = s.supplier_id
    """, conn)

    dim_product.to_csv(f"{output_path}/dim_product.csv", index=False)
    print("✅ dim_product.csv")

except Exception as e:
    print("❌ Product Dim Error:", e)


# 🟣 Customer Dimension (FIXED ❗ removed city)
try:
    dim_customer = pd.read_sql("""
        SELECT customer_id, name, visit_type
        FROM Customer
    """, conn)

    dim_customer.to_csv(f"{output_path}/dim_customer.csv", index=False)
    print("✅ dim_customer.csv")

except Exception as e:
    print("❌ Customer Dim Error:", e)


# 🟣 Date Dimension (FIXED FORMAT 🔥)
try:
    date_range = pd.date_range(start="2023-01-01", end="2025-12-31")

    dim_date = pd.DataFrame({
        'date': date_range,
        'year': date_range.year,
        'month': date_range.month,
        'month_name': date_range.strftime('%B'),
        'day': date_range.day
    })

    # 🔥 IMPORTANT: Ensure same format as fact table
    dim_date['date'] = pd.to_datetime(dim_date['date'])

    dim_date.to_csv(f"{output_path}/dim_date.csv", index=False)
    print("✅ dim_date.csv")

except Exception as e:
    print("❌ Date Dim Error:", e)
# =========================================
# 📊 4. KPI TABLE (FOR CARDS 🔥)
# =========================================
try:
    kpi = pd.read_sql("""
        SELECT 
            SUM(total_amount) AS total_revenue,
            COUNT(sale_id) AS total_orders,
            AVG(total_amount) AS avg_order_value
        FROM Sales
    """, conn)

    kpi.to_csv(f"{output_path}/kpi_summary.csv", index=False)
    print("✅ kpi_summary.csv")

except Exception as e:
    print("❌ KPI Error:", e)

# =========================================
# 🔚 CLOSE
# =========================================
if conn:
    conn.close()
    print("🔒 Connection closed")

print("\n🎉 STAR SCHEMA READY FOR POWER BI")

✅ Connected to MySQL


C:\Users\allab\AppData\Local\Temp\ipykernel_16676\4243827493.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {t}", conn)
C:\Users\allab\AppData\Local\Temp\ipykernel_16676\4243827493.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fact_sales = pd.read_sql("""


✅ Supplier.csv
✅ Product.csv
✅ Inventory.csv
✅ Sales.csv
✅ Sales_Details.csv
✅ Customer.csv
✅ Category.csv
✅ Purchase_Pattern.csv
✅ fact_sales.csv
✅ dim_product.csv
✅ dim_customer.csv
✅ dim_date.csv
✅ kpi_summary.csv
🔒 Connection closed

🎉 STAR SCHEMA READY FOR POWER BI


C:\Users\allab\AppData\Local\Temp\ipykernel_16676\4243827493.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_product = pd.read_sql("""
C:\Users\allab\AppData\Local\Temp\ipykernel_16676\4243827493.py:90: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_customer = pd.read_sql("""
C:\Users\allab\AppData\Local\Temp\ipykernel_16676\4243827493.py:126: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  kpi = pd.read_sql("""
